# 🦕 Stegoceras validum (UALVP 2): Digital Morphology Data Inventory

**Specimen**: *Stegoceras validum* (UALVP 2) — Holotype cranium & skeleton  
**Institution**: University of Alberta Laboratory for Vertebrate Paleontology  
**Phase**: Phase 1 Data Provenance Audit & Ingestion  

---

### 🎯 Objectives
1. Ingest and parse the master YAML dataset manifest (`data/metadata/dataset_manifest.yaml`).
2. Audit all public UALVP 2 digital records according to the **4-tier provenance taxonomy**:
   - `primary_scan`: Raw radiological CT/micro-CT volumes.
   - `segmented_from_primary_scan`: Segmented cranial surface models derived directly from CT.
   - `researcher_derived`: Processed, cleaned, or published benchmark models.
   - `secondary_reference`: Visualizations and educational 3D reconstructions.
3. Inspect local file presence, compute SHA-256 checksums, and provide step-by-step instructions for data ingestion.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import yaml

# Ensure local src is in Python path
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

from stegoceras_biomechanics.io.manifest import load_manifest, audit_local_inventory
from stegoceras_biomechanics.io.ingest import scan_downloads, get_downloads_dir

print(f"Project root: {project_root}")

## 1. Load Master Dataset Manifest
Let's inspect the registered datasets, metadata integrity policy, and provenance levels.

In [ ]:
manifest = load_manifest(project_root / "data" / "metadata" / "dataset_manifest.yaml")
policy = manifest.get("metadata_policy", {})

print("📋 METADATA INTEGRITY POLICY:")
print(f"  • Zero fabrication rule: {policy.get('zero_fabrication')}")
print(f"  • Unverified value fallback: '{policy.get('unverified_values')}'")
print(f"  • Provenance tiers: {policy.get('provenance_levels')}")

# Convert datasets list to DataFrame
df_datasets = pd.DataFrame(manifest.get("datasets", []))
columns_to_show = [
    "dataset_id", "element", "provenance_tier", "repository", 
    "media_id", "license", "download_status"
]
df_datasets[columns_to_show]

## 2. Audit Local Workspace Inventory
Check which datasets currently reside on disk, their sizes, and SHA-256 verification status.

In [ ]:
audit_results = audit_local_inventory(manifest, project_root=project_root)
df_audit = pd.DataFrame(audit_results)

df_audit[["dataset_id", "element", "provenance_tier", "exists", "file_count", "size_bytes", "sha256"]]

## 3. Check Downloads Staging Area
Inspect `data/raw/downloads/` for archive files awaiting ingestion.

In [ ]:
downloads = scan_downloads(project_root=project_root)
print(f"Staging directory: {get_downloads_dir(project_root)}")
if downloads:
    print(f"Found {len(downloads)} file(s) in staging:")
    for d in downloads:
        print(f"  • {d.name} ({d.stat().st_size / (1024*1024):.2f} MB)")
else:
    print("ℹ️ No files currently in downloads staging folder.")

## 4. Instructions for Acquiring Primary CT Data (MorphoSource)

To obtain the raw micro-CT slice stack for *Stegoceras validum* UALVP 2:
1. Visit [MorphoSource Media 000018284](https://www.morphosource.org/concern/media/000018284).
2. Log in with your free MorphoSource user account.
3. Click **Download**, enter your research use statement (minimum 50 characters), and agree to the **CC BY-NC 4.0** license.
4. Save the downloaded archive into `data/raw/downloads/`.
5. Ingest via CLI:
   ```bash
   uv run python scripts/ingest_data.py --scan-downloads
   ```